In [0]:
--2. Compléter la couche Silver:
--Historisez dans le lakehouse en silver
--toutes les tables importées
--précédemment dans la couche bronze.


SHOW TABLES IN barbara_lakehouse.bronze;

DECLARE OR REPLACE load_date = current_timestamp();
VALUES load_date;

-- product
CREATE TABLE IF NOT EXISTS silver.product (
  product_id                  INT,
  name                        STRING,
  product_number              STRING,
  color                       STRING,
  standard_cost               DOUBLE,
  list_price                  DOUBLE,
  size                        STRING,
  weight                      DOUBLE,
  product_category_id         INT,
  product_model_id            INT,
  sell_start_date             TIMESTAMP,
  sell_end_date               TIMESTAMP,
  discontinued_date           TIMESTAMP,
  thumbnail_photo             BINARY,
  thumbnail_photo_file_name   STRING,
  rowguid                     STRING,
  modified_date               TIMESTAMP,

  _tf_valid_from              TIMESTAMP,
  _tf_valid_to                TIMESTAMP,
  _tf_create_date             TIMESTAMP,
  _tf_update_date             TIMESTAMP
)
USING DELTA
;

ALTER TABLE silver.product ADD COLUMNS (
  _tf_valid_from  TIMESTAMP,
  _tf_valid_to    TIMESTAMP,
  _tf_create_date TIMESTAMP,
  _tf_update_date TIMESTAMP
);

MERGE INTO silver.product AS tgt
USING (
    SELECT
        ProductID               AS product_id,
        Name                    AS name,
        ProductNumber           AS product_number,
        Color                   AS color,
        StandardCost            AS standard_cost,
        ListPrice               AS list_price,
        Size                    AS size,
        Weight                  AS weight,
        ProductCategoryID       AS product_category_id,
        ProductModelID          AS product_model_id,
        SellStartDate           AS sell_start_date,
        SellEndDate             AS sell_end_date,
        DiscontinuedDate        AS discontinued_date,
        ThumbNailPhoto          AS thumbnail_photo,
        ThumbnailPhotoFileName  AS thumbnail_photo_file_name,
        rowguid                 AS rowguid,
        ModifiedDate            AS modified_date
    FROM bronze.product
) AS src
ON tgt.ProductID = src.product_id
AND tgt._tf_valid_to IS NULL   -- Only match against 'active' records in silver

WHEN MATCHED AND (
       NOT (tgt.name                     <=> src.name)
    OR NOT (tgt.product_number           <=> src.product_number)
    OR NOT (tgt.color                    <=> src.color)
    OR NOT (tgt.standard_cost            <=> src.standard_cost)
    OR NOT (tgt.list_price               <=> src.list_price)
    OR NOT (tgt.size                     <=> src.size)
    OR NOT (tgt.weight                   <=> src.weight)
    OR NOT (tgt.product_category_id      <=> src.product_category_id)
    OR NOT (tgt.product_model_id         <=> src.product_model_id)
    OR NOT (tgt.sell_start_date          <=> src.sell_start_date)
    OR NOT (tgt.sell_end_date            <=> src.sell_end_date)
    OR NOT (tgt.discontinued_date        <=> src.discontinued_date)
    OR NOT (tgt.thumbnail_photo          <=> src.thumbnail_photo)
    OR NOT (tgt.thumbnail_photo_file_name<=> src.thumbnail_photo_file_name)
    OR NOT (tgt.rowguid                  <=> src.rowguid)
    OR NOT (tgt.modified_date            <=> src.modified_date)
) AND tgt._tf_valid_to IS NULL THEN
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date

WHEN NOT MATCHED BY SOURCE AND tgt._tf_valid_to IS NULL THEN
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date
;

MERGE INTO silver.product AS tgt
USING (
    SELECT
        ProductID               AS product_id,
        Name                    AS name,
        ProductNumber           AS product_number,
        Color                   AS color,
        StandardCost            AS standard_cost,
        ListPrice               AS list_price,
        Size                    AS size,
        Weight                  AS weight,
        ProductCategoryID       AS product_category_id,
        ProductModelID          AS product_model_id,
        SellStartDate           AS sell_start_date,
        SellEndDate             AS sell_end_date,
        DiscontinuedDate        AS discontinued_date,
        ThumbNailPhoto          AS thumbnail_photo,
        ThumbnailPhotoFileName  AS thumbnail_photo_file_name,
        rowguid                 AS rowguid,
        ModifiedDate            AS modified_date
    FROM bronze.product
) AS src
ON tgt.product_id = src.product_id
AND tgt._tf_valid_to IS NULL

WHEN NOT MATCHED THEN
  INSERT (
    product_id,
    name,
    product_number,
    color,
    standard_cost,
    list_price,
    size,
    weight,
    product_category_id,
    product_model_id,
    sell_start_date,
    sell_end_date,
    discontinued_date,
    thumbnail_photo,
    thumbnail_photo_file_name,
    rowguid,
    modified_date,
    _tf_valid_from,
    _tf_valid_to,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.product_id,
    src.name,
    src.product_number,
    src.color,
    src.standard_cost,
    src.list_price,
    src.size,
    src.weight,
    src.product_category_id,
    src.product_model_id,
    src.sell_start_date,
    src.sell_end_date,
    src.discontinued_date,
    src.thumbnail_photo,
    src.thumbnail_photo_file_name,
    src.rowguid,
    src.modified_date,
    load_date,   -- _tf_valid_from
    NULL,        -- _tf_valid_to
    load_date,   -- _tf_create_date
    load_date    -- _tf_update_date
  )
;

-- product category
CREATE TABLE IF NOT EXISTS silver.productcategory (
  product_category_id          INT,
  parent_product_category_id   INT,
  name                         STRING,
  rowguid                      STRING,
  modified_date                TIMESTAMP,

  _tf_valid_from               TIMESTAMP,
  _tf_valid_to                 TIMESTAMP,
  _tf_create_date              TIMESTAMP,
  _tf_update_date              TIMESTAMP
)
USING DELTA
;

MERGE INTO silver.productcategory AS tgt
USING (
    SELECT
        ProductCategoryID        AS product_category_id,
        ParentProductCategoryID  AS parent_product_category_id,
        Name                     AS name,
        rowguid                  AS rowguid,
        ModifiedDate             AS modified_date
    FROM bronze.productcategory
) AS src
ON tgt.product_category_id = src.product_category_id
AND tgt._tf_valid_to IS NULL

WHEN MATCHED AND (
       NOT (tgt.parent_product_category_id <=> src.parent_product_category_id)
    OR NOT (tgt.name                      <=> src.name)
    OR NOT (tgt.rowguid                   <=> src.rowguid)
    OR NOT (tgt.modified_date             <=> src.modified_date)
) AND tgt._tf_valid_to IS NULL THEN
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date

WHEN NOT MATCHED BY SOURCE AND tgt._tf_valid_to IS NULL THEN
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date
;

MERGE INTO silver.productcategory AS tgt
USING (
    SELECT
        ProductCategoryID        AS product_category_id,
        ParentProductCategoryID  AS parent_product_category_id,
        Name                     AS name,
        rowguid                  AS rowguid,
        ModifiedDate             AS modified_date
    FROM bronze.productcategory
) AS src
ON tgt.product_category_id = src.product_category_id
AND tgt._tf_valid_to IS NULL

WHEN NOT MATCHED THEN
  INSERT (
    product_category_id,
    parent_product_category_id,
    name,
    rowguid,
    modified_date,
    _tf_valid_from,
    _tf_valid_to,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.product_category_id,
    src.parent_product_category_id,
    src.name,
    src.rowguid,
    src.modified_date,
    load_date,  -- _tf_valid_from
    NULL,       -- _tf_valid_to
    load_date,  -- _tf_create_date
    load_date   -- _tf_update_date
  )
;
